# Part 5 — Create Image Parquet

Converts JPG images from `data/images/{ASIN}.jpg` into the HuggingFace parquet format
that notebook `02_cluster_centroid_products.ipynb` expects.

**Uses ALL 1,775 ASINs from both train and val splits** — the image parquet is just
a lookup table for cluster visualization, not related to the 80/20 modeling split.

**Output:** `data/amzn_shoes_monthly_avg_long_ffill_28_7_images/`
```
train-00000-of-00004-part000.parquet
train-00000-of-00004-part001.parquet
train-00000-of-00004-part002.parquet
train-00000-of-00004-part003.parquet
train-00001-of-00004-part000.parquet
...  (16 files total)
train-00003-of-00004-part003.parquet
```

## ① Mount Drive

In [1]:
# Drive mount not needed for local execution
print('✅ Local mode')

✅ Local mode


## ② Config

In [2]:
import os
from pathlib import Path

# Detect project root by walking up from CWD to find 'data/' and 'code/' folders
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root (expected 'data/' and 'code/' folders)")

ROOT      = str(PROJECT_ROOT) + os.sep
DATA_DIR  = str(PROJECT_ROOT / 'data') + os.sep
IMG_DIR   = DATA_DIR + 'images' + os.sep
SPLIT_DIR = DATA_DIR + 'amzn_shoes_monthly_diffs_ffill_fixed_splits' + os.sep
import math
import pandas as pd
from pathlib import Path
# This notebook lives in data_preparation/
# All data folders are one level up under demand_modeling_data_women_8/data/
OUT_DIR   = ROOT + 'data/amzn_shoes_monthly_avg_long_ffill_28_7_images/'
# Paper format: 4 shards x 4 parts = 16 parquet files
N_SHARDS = 4
N_PARTS  = 4
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'✅ Output folder ready')
print(f'   {OUT_DIR}')

os.makedirs(DATA_DIR, exist_ok=True)

✅ Output folder ready
   /home/iankuzuma/claude_code/demand_modeling/men-8-subcat-split-separate-embedding/loafers-slip-ons/data/amzn_shoes_monthly_avg_long_ffill_28_7_images/


## ③ Get ALL unique ASINs from both train and val

In [3]:
# Read BOTH splits — 80/20 split is for modeling only
# Here we need ALL ASINs so notebook 02 can look up any product photo
df_train = pd.read_parquet(SPLIT_DIR + 'train-00000-of-00001.parquet')
df_val   = pd.read_parquet(SPLIT_DIR + 'validation-00000-of-00001.parquet')

all_asins = pd.concat([df_train, df_val])['ASIN'].unique().tolist()

print(f'ASINs from train split:   {df_train["ASIN"].nunique()}')
print(f'ASINs from val split:     {df_val["ASIN"].nunique()}')
print(f'Total unique ASINs (all): {len(all_asins)}')

# Check image coverage
asins_with_images = [a for a in all_asins if os.path.exists(IMG_DIR + f'{a}.jpg')]
asins_missing     = [a for a in all_asins if not os.path.exists(IMG_DIR + f'{a}.jpg')]

print(f'\nASINs with images:    {len(asins_with_images)}')
print(f'ASINs missing images: {len(asins_missing)}')
if asins_missing:
    print(f'Missing examples: {asins_missing[:5]}')

ASINs from train split:   180
ASINs from val split:     179
Total unique ASINs (all): 359

ASINs with images:    359
ASINs missing images: 0


## ④ Convert JPGs to parquet (paper format)

In [4]:
total_asins = len(all_asins)
total_files = N_SHARDS * N_PARTS  # 16 files

# Split all ASINs evenly into 16 chunks
chunk_size = math.ceil(total_asins / total_files)
chunks = [all_asins[i:i+chunk_size] for i in range(0, total_asins, chunk_size)]

# Pad to exactly 16 if needed
while len(chunks) < total_files:
    chunks.append([])
chunks = chunks[:total_files]

print(f'Total ASINs:         {total_asins}')
print(f'Total parquet files: {total_files}')
print(f'ASINs per file:      ~{chunk_size}')
print()

file_idx = 0
for shard in range(N_SHARDS):
    for part in range(N_PARTS):
        chunk_asins = chunks[file_idx]
        file_idx += 1

        rows = []
        for asin in chunk_asins:
            img_path = IMG_DIR + f'{asin}.jpg'
            if os.path.exists(img_path):
                with open(img_path, 'rb') as f:
                    img_bytes = f.read()
                image_dict = {'bytes': img_bytes, 'path': None}
            else:
                image_dict = {'bytes': b'', 'path': None}

            rows.append({'ASIN': asin, 'image': image_dict})

        if not rows:
            continue

        df_chunk = pd.DataFrame(rows)
        fname    = f'train-0000{shard}-of-00004-part00{part}.parquet'
        out_path = OUT_DIR + fname
        df_chunk.to_parquet(out_path, index=False)

        size_mb = os.path.getsize(out_path) / 1024 / 1024
        print(f'✅ {fname}  ({len(rows)} ASINs, {size_mb:.1f} MB)')

print(f'\n🎉 Done! All 16 parquet files saved.')

Total ASINs:         359
Total parquet files: 16
ASINs per file:      ~23

✅ train-00000-of-00004-part000.parquet  (23 ASINs, 3.8 MB)
✅ train-00000-of-00004-part001.parquet  (23 ASINs, 4.5 MB)
✅ train-00000-of-00004-part002.parquet  (23 ASINs, 4.9 MB)
✅ train-00000-of-00004-part003.parquet  (23 ASINs, 5.5 MB)
✅ train-00001-of-00004-part000.parquet  (23 ASINs, 5.6 MB)


✅ train-00001-of-00004-part001.parquet  (23 ASINs, 3.8 MB)
✅ train-00001-of-00004-part002.parquet  (23 ASINs, 4.7 MB)
✅ train-00001-of-00004-part003.parquet  (23 ASINs, 4.9 MB)


✅ train-00002-of-00004-part000.parquet  (23 ASINs, 4.5 MB)
✅ train-00002-of-00004-part001.parquet  (23 ASINs, 4.7 MB)
✅ train-00002-of-00004-part002.parquet  (23 ASINs, 6.0 MB)
✅ train-00002-of-00004-part003.parquet  (23 ASINs, 5.0 MB)
✅ train-00003-of-00004-part000.parquet  (23 ASINs, 5.8 MB)
✅ train-00003-of-00004-part001.parquet  (23 ASINs, 3.8 MB)
✅ train-00003-of-00004-part002.parquet  (23 ASINs, 4.0 MB)
✅ train-00003-of-00004-part003.parquet  (14 ASINs, 2.4 MB)

🎉 Done! All 16 parquet files saved.


## ⑤ Verify — load back exactly as notebook 02 does

In [5]:
import io
import datasets
from PIL import Image

# Load exactly as notebook 02 does
data_files = {
    'train': [f'train-0000{i}-of-00004-part00{j}.parquet' for i in range(4) for j in range(4)],
}

ds_img = datasets.load_dataset(
    'parquet',
    data_dir=OUT_DIR,
    data_files=data_files,
)

df_img = ds_img['train'].to_pandas()
df_img = df_img.set_index('ASIN')

print(f'Total ASINs loaded: {len(df_img)}')
print(f'Columns: {list(df_img.columns)}')

# Test reading one image exactly as notebook 02 does
test_asin = df_img.index[0]
img = Image.open(io.BytesIO(df_img.loc[test_asin, :].values[0]['bytes']))
print(f'\n✅ Test image OK — ASIN: {test_asin}, Size: {img.size}, Mode: {img.mode}')
print('\n✅ Ready for notebook 02!')

Generating train split: 0 examples [00:00, ? examples/s]

Total ASINs loaded: 359
Columns: ['image']

✅ Test image OK — ASIN: B0007T5JCI, Size: (1920, 1197), Mode: RGB

✅ Ready for notebook 02!
